# CardioIA - Ir Alem 2\n\nComparacao entre regressao logistica e um modelo neuromorfico LIF simples para series temporais sinteticas de batimentos cardiacos.

In [ ]:
import numpy as np\nimport pandas as pd\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import accuracy_score, classification_report, confusion_matrix\nfrom sklearn.model_selection import train_test_split\nimport matplotlib.pyplot as plt\n\nrng = np.random.default_rng(57)

## Geracao de dados sinteticos\n\nCada amostra possui 60 pontos. A classe 0 representa ritmo normal e a classe 1 representa risco, com maior variabilidade e picos.

In [ ]:
def generate_series(n_samples=400, length=60):\n    x = []\n    y = []\n    t = np.linspace(0, 2 * np.pi, length)\n    for i in range(n_samples):\n        label = i >= n_samples // 2\n        base = 75 + 5 * np.sin(t * 3) + rng.normal(0, 2, length)\n        if label:\n            spikes = rng.choice(length, size=5, replace=False)\n            base += rng.normal(8, 5, length)\n            base[spikes] += rng.normal(35, 8, len(spikes))\n        x.append(base)\n        y.append(int(label))\n    return np.array(x), np.array(y)\n\nX, y = generate_series()\nplt.figure(figsize=(10, 4))\nplt.plot(X[y == 0][0], label='normal')\nplt.plot(X[y == 1][0], label='risco')\nplt.legend();\nplt.title('Series sinteticas de batimentos');

## Classificador tradicional\n\nA regressao logistica usa features estatisticas simples, mantendo interpretabilidade.

In [ ]:
def extract_features(series):\n    peaks = (series > 110).sum(axis=1)\n    return np.column_stack([\n        series.mean(axis=1),\n        series.std(axis=1),\n        series.max(axis=1),\n        series.min(axis=1),\n        series.max(axis=1) - series.min(axis=1),\n        peaks,\n    ])\n\nfeatures = extract_features(X)\nX_train, X_test, y_train, y_test = train_test_split(features, y, test_size=0.25, random_state=57, stratify=y)\nclf = LogisticRegression(max_iter=1000)\nclf.fit(X_train, y_train)\npred_lr = clf.predict(X_test)\nprint('Acuracia regressao logistica:', accuracy_score(y_test, pred_lr))\nprint(confusion_matrix(y_test, pred_lr))\nprint(classification_report(y_test, pred_lr, target_names=['normal', 'risco']))

## Modelo neuromorfico LIF simples\n\nO modelo Leaky Integrate-and-Fire acumula potencial, aplica vazamento e conta disparos quando o limiar e ultrapassado.

In [ ]:
def lif_spike_count(series, threshold=1.0, leak=0.85, scale=35.0):\n    centered = (series - 70) / scale\n    counts = []\n    for row in centered:\n        v = 0.0\n        spikes = 0\n        for value in row:\n            v = leak * v + max(value, 0)\n            if v >= threshold:\n                spikes += 1\n                v = 0.0\n        counts.append(spikes)\n    return np.array(counts)\n\nX_train_s, X_test_s, y_train_s, y_test_s = train_test_split(X, y, test_size=0.25, random_state=57, stratify=y)\ntrain_spikes = lif_spike_count(X_train_s)\ntest_spikes = lif_spike_count(X_test_s)\n\ncandidate_thresholds = np.arange(train_spikes.min(), train_spikes.max() + 1)\nbest_threshold = max(candidate_thresholds, key=lambda th: accuracy_score(y_train_s, train_spikes >= th))\npred_lif = (test_spikes >= best_threshold).astype(int)\n\nprint('Limiar LIF escolhido:', best_threshold)\nprint('Acuracia LIF:', accuracy_score(y_test_s, pred_lif))\nprint(confusion_matrix(y_test_s, pred_lif))\nprint(classification_report(y_test_s, pred_lif, target_names=['normal', 'risco']))

## Conclusao\n\nA regressao logistica e mais direta para bases tabulares pequenas e fornece pesos interpretaveis. O LIF e interessante como conceito neuromorfico, pois converte dinamica temporal em spikes, mas depende de calibracao cuidadosa de limiar, vazamento e escala. Em uma solucao real da CardioIA, ambos exigiriam validacao com sinais clinicos reais e revisao medica.